In [ ]:
from torch import nn
import timm
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import pandas as pd
import os
from timm import create_model
import seaborn as sns

## Load Model

In [ ]:
class HRNetCustomModel(nn.Module):
        def __init__(self):
                super(HRNetCustomModel, self).__init__()
                
                # HRNet Backbone (ใช้จาก timm)
                self.hrnet = timm.create_model('hrnet_w48', pretrained=True, features_only=True)
                
                # Additional Fully Connected Layers
                self.flatten = nn.Flatten()
                self.fc1 = nn.Linear(1024 * 16 * 16, 1024)  # ปรับขนาดให้ตรงกับ output feature map
                self.fc2 = nn.Linear(1024, 512)
                self.fc3 = nn.Linear(512, 256)
                self.fc4 = nn.Linear(256, 128)
                self.fc5 = nn.Linear(128, 64)
                self.fc6 = nn.Linear(64, 3)
        def forward(self, x):
                features = self.hrnet(x)
                x = features[-1]  # ใช้ feature map สุดท้าย
                x = self.flatten(x)
                x = torch.relu(self.fc1(x))
                x = torch.relu(self.fc2(x))
                x = torch.relu(self.fc3(x))
                x = torch.relu(self.fc4(x))
                x = torch.relu(self.fc5(x))
                return self.fc6(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rotation_model = HRNetCustomModel().to(device)
rotation_model.load_state_dict(torch.load('/Users/paritt.w/Desktop/STRAIGHT/Model/best_HR+reg3(0.870).pth', weights_only=True,map_location=device))
rotation_model.to(device)
rotation_model.eval()

In [ ]:
classification_model_weights_path = '/Users/paritt.w/Desktop/STRAIGHT/Model/Best_classification3.pth'
classification_model = create_model("hrnet_w48", pretrained=True, num_classes=3)
classification_model.load_state_dict(torch.load(classification_model_weights_path, map_location=device))
classification_model.to(device)
classification_model.eval()

## Analyze

In [ ]:
def np_to_torch(np_array):      return torch.from_numpy(np_array).float()
def torch_to_np(torch_array):   return np.squeeze(torch_array.detach().cpu().numpy())
def preprocess(img):
        preprocessing_fn = smp.encoders.get_preprocessing_fn('resnext50_32x4d', 'imagenet')
        img_r = img.astype(np.float64)
        img = preprocessing_fn(img)
        image = img.astype(np.float64)
        image = np.transpose(image, (2, 0, 1))
        img_input = np.expand_dims(image, 0)
        img_input = np_to_torch(img_input).to(device)
        
        image_r = np.transpose(img_r, (2, 0, 1))
        img_input_r = np.expand_dims(image_r, 0)
        img_input_r = np_to_torch(img_input_r).to(device)
        return img_input, img_input_r

def cal_distance(point):
        x1, x2, x3 = point
        distance_left = round(x2 - x3)
        distance_right = round(x3 - x1)
        return distance_left, distance_right

def cal_alpha(distance_left, distance_right):
        alpha = (distance_right - distance_left) / (distance_right + distance_left)
        return round(alpha, 3)

def straight_pred(imagePath, alpha_threshold=0.2):
        SIZE_X = 512 
        SIZE_Y = 512
        image = cv2.imread(imagePath, 1)
        image = cv2.resize(image, (SIZE_Y, SIZE_X), interpolation = cv2.INTER_NEAREST)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image/255
        img_input, img_input_r = preprocess(image)
        
        rotate_pred = rotation_model(img_input_r)
        rotate_pred = rotate_pred[0, :].detach().cpu().numpy()
        distance_left, distance_right = cal_distance(rotate_pred)
        alpha = cal_alpha(distance_left, distance_right)
        if alpha > alpha_threshold:
                rotation_class = "Lt Rotate"
        elif alpha < -alpha_threshold:
                rotation_class = "Rt Rotate"
        else:
                rotation_class = "No Rotation"        
        return rotate_pred, distance_left, distance_right, alpha, rotation_class

        
def classify_model_pred(imagePath):
        SIZE_X = 512 
        SIZE_Y = 512
        image = cv2.imread(imagePath, 1)
        image = cv2.resize(image, (SIZE_Y, SIZE_X), interpolation = cv2.INTER_NEAREST)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image / 255.0
        
        image_tensor = np.transpose(image, (2, 0, 1))
        image_tensor = np.expand_dims(image_tensor, 0)
        image_tensor = torch.from_numpy(image_tensor).float().to(device)
        
        output = classification_model(image_tensor)
        probs = torch.softmax(output, dim=1).detach().cpu().numpy()[0]
        _, preds = torch.max(output, 1)
        class_names = ["Lt Rotate", "No Rotation", "Rt Rotate"]
        return class_names[preds[0].item()], probs

def ground_truth_pred(imagePath, dataset, alpha_threshold=0.2):
        filename = os.path.basename(imagePath)
        gt_row = dataset[dataset['Image'] == filename]
        gt_x1 = gt_row['X1'].values[0]
        gt_x2 = gt_row['X2'].values[0]
        gt_x3 = gt_row['X3'].values[0]
        distance_left, distance_right = cal_distance((gt_x1, gt_x2, gt_x3))
        alpha = cal_alpha(distance_left, distance_right)
        if alpha > alpha_threshold:
                rotation_class = "Lt Rotate"
        elif alpha < -alpha_threshold:
                rotation_class = "Rt Rotate"
        else:
                rotation_class = "No Rotation"                
        return (gt_x1, gt_x2, gt_x3), distance_left, distance_right, alpha, rotation_class

def gt_pred(gt_x1, gt_x2, gt_x3, alpha_threshold=0.2):
        distance_left, distance_right = cal_distance((gt_x1, gt_x2, gt_x3))
        alpha = cal_alpha(distance_left, distance_right)
        if alpha > alpha_threshold:
                rotation_class = "Lt Rotate"
        elif alpha < -alpha_threshold:
                rotation_class = "Rt Rotate"
        else:
                rotation_class = "No Rotation"                
        return distance_left, distance_right, alpha, rotation_class

In [ ]:
data = pd.read_csv('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/Test.csv')
gt_pred_list = []
straight_pred_list = []
class_pred_list = []
gt_x1_list, gt_x2_list, gt_x3_list = [], [], []
st_x1_list, st_x2_list, st_x3_list = [], [], []
gt_alpha_list = []
st_alpha_list = []
cl_prob_list = []

st_wrong_list = []
cl_wrong_list = []
both_wrong_list = []
both_right_list = []

for i in range(len(data['Image'])):
    path = '/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + data['Image'][i]
    gt_x1, gt_x2, gt_x3 = data['X1'][i], data['X2'][i], data['X3'][i]
    gt_distance_left, gt_distance_right, gt_alpha, gt_pred_class = gt_pred(gt_x1, gt_x2, gt_x3)
    rotate_pred, distance_left, distance_right, alpha, str_pred_class = straight_pred(path)
    class_pred_class, class_probs = classify_model_pred(path)
    gt_pred_list.append(gt_pred_class)
    straight_pred_list.append(str_pred_class)
    class_pred_list.append(class_pred_class)
    gt_alpha_list.append(gt_alpha)
    st_alpha_list.append(alpha)
    cl_prob_list.append(class_probs)
    gt_x1_list.append(gt_x1)
    gt_x2_list.append(gt_x2)
    gt_x3_list.append(gt_x3)
    st_x1_list.append(rotate_pred[0])
    st_x2_list.append(rotate_pred[1])
    st_x3_list.append(rotate_pred[2])
    if str_pred_class != gt_pred_class and class_pred_class == gt_pred_class:
        st_wrong_list.append(data['Image'][i])
    elif str_pred_class == gt_pred_class and class_pred_class != gt_pred_class:
        cl_wrong_list.append(data['Image'][i])
    elif str_pred_class != gt_pred_class and class_pred_class != gt_pred_class:
        both_wrong_list.append(data['Image'][i])
    elif str_pred_class == gt_pred_class and class_pred_class == gt_pred_class:
        both_right_list.append(data['Image'][i])


In [ ]:
def cal_MAE(gt_alpha_list, pred_alpha_list):
    absolute_errors = [abs(gt - pred) for gt, pred in zip(gt_alpha_list, pred_alpha_list)]
    mae = sum(absolute_errors) / len(absolute_errors)
    return mae

In [ ]:
print("MAE of STRAIGHT X1:", round(cal_MAE(gt_x1_list, st_x1_list), 2))
print("MAE of STRAIGHT X2:", round(cal_MAE(gt_x2_list, st_x2_list), 2))
print("MAE of STRAIGHT X3:", round(cal_MAE(gt_x3_list, st_x3_list), 2))
print("Average MAE of STRAIGHT X1, X2, X3:", round((cal_MAE(gt_x1_list, st_x1_list) + cal_MAE(gt_x2_list, st_x2_list) + cal_MAE(gt_x3_list, st_x3_list)) / 3, 2))
print("MAE of STRAIGHT Alpha:", round(cal_MAE(gt_alpha_list, st_alpha_list), 2))

# ── mm conversion ──────────────────────────────────────────────────────────
# DICOM pixel spacing at original acquisition resolution, and that original
# size, are constant across the dataset, so a single scale factor applies.
PIXEL_SPACING_MM = 0.143   # mm/px at original DICOM resolution
ORIGINAL_SIZE_PX = 2992    # original image width/height before resizing
RESIZED_SIZE_PX  = 512     # resolution X1/X2/X3 predictions are made at
MM_PER_PX = PIXEL_SPACING_MM * (ORIGINAL_SIZE_PX / RESIZED_SIZE_PX)

print(f"\nPixel spacing at {RESIZED_SIZE_PX}x{RESIZED_SIZE_PX}: {MM_PER_PX:.4f} mm/px")
print("MAE of STRAIGHT X1 (mm):", round(cal_MAE(gt_x1_list, st_x1_list) * MM_PER_PX, 2))
print("MAE of STRAIGHT X2 (mm):", round(cal_MAE(gt_x2_list, st_x2_list) * MM_PER_PX, 2))
print("MAE of STRAIGHT X3 (mm):", round(cal_MAE(gt_x3_list, st_x3_list) * MM_PER_PX, 2))
print("Average MAE of STRAIGHT X1, X2, X3 (mm):", round((cal_MAE(gt_x1_list, st_x1_list) + cal_MAE(gt_x2_list, st_x2_list) + cal_MAE(gt_x3_list, st_x3_list)) / 3 * MM_PER_PX, 2))

In [ ]:
from sklearn.metrics import confusion_matrix

class_names = ["Rt Rotate", "No Rotation", "Lt Rotate"]
class_to_idx = {c: i for i, c in enumerate(class_names)}

y_true = np.array([class_to_idx[c] for c in gt_pred_list])
y_straight = np.array([class_to_idx[c] for c in straight_pred_list])
y_classify = np.array([class_to_idx[c] for c in class_pred_list])
    
# Confusion matrices
cm_straight = confusion_matrix(y_true, y_straight, labels=[0, 1, 2])
cm_classify = confusion_matrix(y_true, y_classify, labels=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    cm_straight,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0],
)
axes[0].set_title("STRAIGHT", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Ground Truth")

sns.heatmap(
    cm_classify,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1],
)
axes[1].set_title("Classification Model", fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Ground Truth")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

# Metrics helpers
def _specificity_macro(y_t, y_p, n_classes=3):
    cm = confusion_matrix(y_t, y_p, labels=list(range(n_classes)))
    specs = []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        denom = tn + fp
        specs.append(tn / denom if denom > 0 else np.nan)
    return float(np.nanmean(specs))

def _compute_metrics(y_t, y_p):
    return {
        "Sensitivity": recall_score(y_t, y_p, average="macro", zero_division=0),
        "Specificity": _specificity_macro(y_t, y_p, n_classes=3),
        "Precision": precision_score(y_t, y_p, average="macro", zero_division=0),
        "Accuracy": accuracy_score(y_t, y_p),
        "F1": f1_score(y_t, y_p, average="macro", zero_division=0),
    }

metrics_straight = _compute_metrics(y_true, y_straight)
metrics_classify = _compute_metrics(y_true, y_classify)
metrics_human = {
        "Sensitivity": 0.64,
        "Specificity": 0.97,
        "Precision": 0.89,
        "Accuracy": 0.86,
        "F1": 0.70,
    }

# Summary table
summary_df = pd.DataFrame(
    {
        "Model": ["STRAIGHT", "Classification", "Human"],
        "Sensitivity": [metrics_straight["Sensitivity"], metrics_classify["Sensitivity"], metrics_human["Sensitivity"]],
        "Specificity": [metrics_straight["Specificity"], metrics_classify["Specificity"], metrics_human["Specificity"]],
        "Precision": [metrics_straight["Precision"], metrics_classify["Precision"], metrics_human["Precision"]],
        "Accuracy": [metrics_straight["Accuracy"], metrics_classify["Accuracy"], metrics_human["Accuracy"]],
        "F1": [metrics_straight["F1"], metrics_classify["F1"], metrics_human["F1"]]
    }
)

display(summary_df.style.format({
    "Sensitivity": "{:.2f}",
    "Specificity": "{:.2f}",
    "Precision": "{:.2f}",
    "Accuracy": "{:.2f}",
    "F1": "{:.2f}"
}))

In [ ]:
metrics_names = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']
human_values = [0.86,0.64,0.97,0.89,0.70]
straight_values = [metrics_straight[m] for m in metrics_names]
classification_values = [metrics_classify[m] for m in metrics_names]
    
# Number of variables
num_vars = len(metrics_names)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# Complete the circle
human_values += human_values[:1]
straight_values += straight_values[:1]
classification_values += classification_values[:1]
angles += angles[:1]

pack_radar = (angles, human_values, straight_values, classification_values)
# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'), dpi=300)

# Plot data
ax.plot(angles, straight_values, 'o-', linewidth=2, label='STRAIGHT', color="#1100FF")
ax.fill(angles, straight_values, alpha=0.15, color="#1100FF")

ax.plot(angles, classification_values, 'o-', linewidth=2, label='CNN', color="#4BC86D")
ax.fill(angles, classification_values, alpha=0.15, color="#4BC86D")

ax.plot(angles, human_values, 'o-', linewidth=2, label='Human Avg', color="#FF0000")
ax.fill(angles, human_values, alpha=0.15, color="#FF0000")

# Fix axis to go in the right order and start at 12 o'clock
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Draw axis lines for each angle and label
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, size=12, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=10)

# Add legend and title
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

# Add grid
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# ============================
# Bootstrap AUC Confidence Intervals
# ============================
def bootstrap_auc(y_true, y_scores, n_bootstraps=10**5, confidence_level=0.95, random_state=42):
    """
    Calculate bootstrap confidence interval for AUC.
    
    Args:
        y_true: True labels
        y_scores: Prediction scores
        n_bootstraps: Number of bootstrap iterations
        confidence_level: Confidence level (default 0.95 for 95% CI)
        random_state: Random seed for reproducibility
        
    Returns:
        auc_value: Original AUC
        ci_lower: Lower bound of CI
        ci_upper: Upper bound of CI
    """
    np.random.seed(random_state)
    
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    n_samples = len(y_true)
    
    # Calculate original AUC
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    auc_value = auc(fpr, tpr)
    
    # Bootstrap
    bootstrapped_aucs = []
    for i in range(n_bootstraps):
        # Resample with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Skip if all samples are same class
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        # Calculate AUC for this bootstrap sample
        fpr_boot, tpr_boot, _ = roc_curve(y_true[indices], y_scores[indices])
        auc_boot = auc(fpr_boot, tpr_boot)
        bootstrapped_aucs.append(auc_boot)
    
    # Calculate confidence intervals
    alpha = 1 - confidence_level
    ci_lower = np.percentile(bootstrapped_aucs, 100 * alpha / 2)
    ci_upper = np.percentile(bootstrapped_aucs, 100 * (1 - alpha / 2))
    
    return auc_value, ci_lower, ci_upper

In [ ]:
abs_gt_alpha = np.abs(np.array(gt_alpha_list))
abs_st_alpha = np.abs(np.array(st_alpha_list))
y_true_binary = (abs_gt_alpha > 0.2).astype(int)
y_scores_straight = abs_st_alpha
y_scores_classify = [max(probs[0], probs[2]) for probs in cl_prob_list]

In [ ]:
# Calculate AUC with 95% CI for both methods
auc_str, auc_str_lower, auc_str_upper = bootstrap_auc(y_true_binary, y_scores_straight)
auc_clf, auc_clf_lower, auc_clf_upper = bootstrap_auc(y_true_binary, y_scores_classify)

In [ ]:
fpr_straight, tpr_straight, _ = roc_curve(y_true_binary, y_scores_straight)
fpr_classification, tpr_classification, _ = roc_curve(y_true_binary, y_scores_classify)

In [ ]:
# Plot ROC curves
roc_pack = (fpr_straight, tpr_straight, auc_str,
            fpr_classification, tpr_classification, auc_clf)
fig, ax = plt.subplots(1, 1, figsize=(10, 8), dpi=300)

ax.plot(fpr_straight, tpr_straight, 'black', linestyle='-', linewidth=2,
        label=f'STRAIGHT\nAUC: {auc_str:.2f} ({auc_str_lower:.2f}, {auc_str_upper:.2f})')
ax.plot(fpr_classification, tpr_classification, 'black', linestyle='--',linewidth=2,
        label=f'Classification CNN\nAUC: {auc_clf:.2f} ({auc_clf_lower:.2f}, {auc_clf_upper:.2f})')
# ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier (AUC = 0.5000)')

ax.set_xlabel('1 - Specificity', fontsize=12)
ax.set_ylabel('Sensitivity', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
# ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

# Remove top and right spines (box)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


def _get_last_conv_layer(model):
    """Return the deepest Conv2d layer for Grad-CAM target."""
    last_conv = None
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError("No Conv2d layer found in the model.")
    return last_conv


def pred_and_cam(imagePath, model=classification_model, target_layer=None, target_class=None, use_cutoff=False, cutoff_threshold=0.5):
    # Load and preprocess image
    image = cv2.imread(imagePath)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_float = image.astype(np.float32) / 255.0
    input_tensor = torch.from_numpy(np.transpose(image_float, (2, 0, 1))).unsqueeze(0).float().to(device)

    # Get prediction
    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()
        confidence = torch.softmax(output, dim=1)[0, predicted_class].item()

    class_names = ["Lt Rotate", "No Rotation", "Rt Rotate"]
    class_map = {name: idx for idx, name in enumerate(class_names)}
    print(f"Predicted class: {list(class_map.keys())[list(class_map.values()).index(predicted_class)]} with confidence: {confidence:.4f}")

    # Use the last convolutional layer by default (works for HRNet and other CNN backbones)
    if target_layer is None:
        target_layer = _get_last_conv_layer(model)

    # Set target class (default to predicted class)
    if target_class is None:
        target_class = predicted_class

    # Initialize GradCAM
    cam = GradCAM(model=model, target_layers=[target_layer])

    # Generate GradCAM
    grayscale_cam = cam(input_tensor=input_tensor, targets=None if target_class == predicted_class else [target_class])
    grayscale_cam = grayscale_cam[0, :]

    # Apply cutoff if enabled
    if use_cutoff:
        print(f"Applying cutoff threshold: {cutoff_threshold} (min: {grayscale_cam.min():.3f}, max: {grayscale_cam.max():.3f})")
        # Create mask for values below threshold
        mask = grayscale_cam < cutoff_threshold
        grayscale_cam = np.where(mask, 0, grayscale_cam)
        # Renormalize after cutoff (only non-zero values)
        if grayscale_cam.max() > 0:
            grayscale_cam = (grayscale_cam - grayscale_cam.min()) / (grayscale_cam.max() - grayscale_cam.min())

    # Create visualization with transparency for cutoff regions
    if use_cutoff:
        # Create custom overlay with transparency
        from matplotlib import cm

        colormap = cm.get_cmap("jet")
        heatmap = colormap(grayscale_cam)[:, :, :3]  # RGB only

        # Create alpha channel based on grayscale_cam (0 where cam is 0, 1 where cam > 0)
        alpha = (grayscale_cam > 0).astype(float) * 0.5  # 0.5 opacity for overlay

        # Blend: show original image where alpha is 0, overlay where alpha > 0
        visualization = image_float.copy()
        for i in range(3):  # RGB channels
            visualization[:, :, i] = image_float[:, :, i] * (1 - alpha) + heatmap[:, :, i] * alpha
        visualization = (visualization * 255).astype(np.uint8)
    else:
        # Use default show_cam_on_image for non-cutoff mode
        visualization = show_cam_on_image(image_float, grayscale_cam, use_rgb=True)

    return predicted_class, confidence, visualization

In [ ]:
def overlay_straight_pred(imagePath, pred_x1, pred_x2, pred_x3):
    image = cv2.imread(imagePath)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    overlay = image.copy()
    alpha = 0.7  # Transparency factor

    # Draw vertical lines for X1, X2, X3
    for x in [pred_x1, pred_x2, pred_x3]:
        cv2.line(overlay, (int(x), 0), (int(x), overlay.shape[0]), (255, 255, 255), thickness=2)

    # Blend original image with overlay
    blended = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)
    return blended

In [ ]:
def display_predictions(image_path, save_dir=None, save=False, gt_label=None):
    image_name = os.path.basename(image_path)
    original_image = cv2.imread(image_path)
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    gt_class = gt_label if gt_label is not None else ground_truth_pred(image_path, data)[4]
    rotate_pred, distance_left, distance_right, alpha, str_pred_class = straight_pred(image_path)
    overlay_img = overlay_straight_pred(image_path, rotate_pred[0], rotate_pred[1], rotate_pred[2])
    predicted_class, confidence, visualization = pred_and_cam(image_path, model=classification_model, use_cutoff=True, cutoff_threshold=0.5)
    percent_confidence = confidence * 100

    class_names = ["Lt Rotate", "No Rotation", "Rt Rotate"]
    predicted_class_name = class_names[predicted_class]

    # Escape math-text special chars in file name so bold title renders correctly.
    safe_image_name = image_name.replace("\\", r"\\").replace("_", r"\_").replace("%", r"\%")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=300)
    axes[0].imshow(original_image)
    axes[0].set_title(rf"$\bf{{{safe_image_name}}}$" + f"\nGT: {gt_class}")
    axes[0].axis('off')
    axes[1].imshow(overlay_img)
    axes[1].set_title(r"$\bf{STRAIGHT}$" + f"\nPred: {str_pred_class} (Alpha: {alpha:.2f})")
    axes[1].axis('off')
    axes[2].imshow(visualization)
    axes[2].set_title(r"$\bf{Classification\ CNN}$" + f"\nPred: {predicted_class_name} (%Confidence: {percent_confidence:.2f}%)")
    axes[2].axis('off')
    plt.tight_layout(pad=2.5)
    if save and save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        base_name = os.path.basename(image_path).replace('.png', '')
        save_path = os.path.join(save_dir, f"{base_name}_comparison.png")
        fig.savefig(save_path, dpi=300)
    plt.show()

## STRAIGHT vs Classifier

In [ ]:
print('=' * 50)
print("STRAIGHT vs Classification Model")
print('=' * 50)
print(f"STRAIGHT wrong, Classification right: {len(st_wrong_list)}")
print(f"STRAIGHT right, Classification wrong: {len(cl_wrong_list)}")
print(f"Both wrong: {len(both_wrong_list)}")
print(f"Both right: {len(both_right_list)}")
print(f"Total samples: {len(gt_pred_list)}")

### Custom image compare

In [ ]:
img_path = '/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Custom image/A_batch1_140.png'
save_dir = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/Custom'
gt_label = '??'
display_predictions(img_path, save_dir=save_dir, save=True, gt_label=gt_label)

In [ ]:
img_path = '/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Custom image/A_batch4_1810.png'
save_dir = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/Custom'
gt_label = '??'
display_predictions(img_path, save_dir=save_dir, save=True, gt_label=gt_label)

### STRAIGHT wrong, Classifier right

In [ ]:
save_folder_name = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/STRAIGHT_wrong_classification_right'
for img in st_wrong_list:
    display_predictions('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + img, save_dir=save_folder_name, save=True)

### STRAIGHT right, Classifier wrong

In [ ]:
for img in cl_wrong_list:
    display_predictions('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + img)

### STRAIGHT right, Classifier wrong

In [ ]:
save_folder_name = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/STRAIGHT_right_classification_wrong'
for img in cl_wrong_list:
    display_predictions('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + img, save_dir=save_folder_name, save=True)

### Both wrong

In [ ]:
save_folder_name = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/Both_wrong'
for img in both_wrong_list:
    display_predictions('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + img, save_dir=save_folder_name, save=True)

### Both right

In [ ]:
save_folder_name = '/Users/paritt.w/Desktop/STRAIGHT/Fig/Analysis_Results/Both_right'
for img in both_right_list:
    display_predictions('/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/' + img, save_dir=save_folder_name, save=True)